In [ ]:
# LifeCost — 2026 Data Model & Calculation Dataset

This notebook builds the new production-ready data layer for LifeCost.

## Data sources

1. Education & university course dataset
2. Australia city cost-of-living dataset

## Objectives

- Load the new raw datasets
- Validate course and university data
- Validate all 8 cities
- Standardize field names and values
- Verify course + city + university relationships
- Verify tuition and duration fields
- Prepare clean datasets for the LifeCost calculation engine
- Generate data-quality reports

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Current working directory:")
print(Path.cwd())

print("\nRaw data directory:")
print(RAW_DIR.resolve())

print("\nRaw directory exists:", RAW_DIR.exists())

Current working directory:
/Users/vedantburgul/Desktop/LifeCost Project/notebooks

Raw data directory:
/Users/vedantburgul/Desktop/LifeCost Project/data/raw

Raw directory exists: True


In [10]:
education_candidates = sorted(
    RAW_DIR.glob("education_courses_2026_v2*.csv")
)

living_candidates = sorted(
    RAW_DIR.glob("australia_cost_of_living_2026_v2*.csv")
)

print("Education candidates:")
for f in education_candidates:
    print(" -", f.name)

print("\nLiving-cost candidates:")
for f in living_candidates:
    print(" -", f.name)

Education candidates:
 - education_courses_2026_v2..csv

Living-cost candidates:
 - australia_cost_of_living_2026_v2.csv


In [11]:
EDUCATION_FILE = education_candidates[0]
LIVING_FILE = living_candidates[0]

education_raw = pd.read_csv(EDUCATION_FILE)
living_raw = pd.read_csv(LIVING_FILE)

print("Education file loaded:")
print(EDUCATION_FILE.name)

print("\nLiving-cost file loaded:")
print(LIVING_FILE.name)

print("\nEducation shape:", education_raw.shape)
print("Living-cost shape:", living_raw.shape)

Education file loaded:
education_courses_2026_v2..csv

Living-cost file loaded:
australia_cost_of_living_2026_v2.csv

Education shape: (69, 14)
Living-cost shape: (8, 56)


In [12]:
print("EDUCATION DATASET")
print("=" * 60)

print("Columns:")
for i, col in enumerate(education_raw.columns, start=1):
    print(f"{i}. {col}")

print("\nFirst 5 rows:")
display(education_raw.head())

EDUCATION DATASET
Columns:
1. City
2. University
3. Course
4. Available
5. International_Student
6. Campus
7. Duration_Years
8. Annual_Tuition_AUD
9. Total_Tuition_AUD
10. CRICOS_Code
11. Start_Dates
12. Tuition_Source
13. Course_Source
14. Last_Verified

First 5 rows:


,City,University,Course,Available,International_Student,Campus,Duration_Years,Annual_Tuition_AUD,Total_Tuition_AUD,CRICOS_Code,Start_Dates,Tuition_Source,Course_Source,Last_Verified
0,Melbourne,RMIT University,Master of Data Science,Yes,Yes,Melbourne City,2,"A$44,160","A$88,320",093313B,February / July,RMIT 2026 international course page,RMIT official course page,2026-08-20
1,Melbourne,The University of Melbourne,Master of Data Science,Yes,Yes,Parkville,2,"A$57,984","A$115,968",092791B,March,University of Melbourne 2026 international fees,University of Melbourne official course page,2026-08-20
2,Melbourne,Monash University,Master of Data Science,Yes,Yes,Clayton,2,"A$52,900","A$105,800",085349A,February / July,Monash 2026 international guide,Monash official course page,2026-08-20
3,Melbourne,La Trobe University,Master of Data Science,Yes,Yes,Bundoora,2,"A$43,800","A$87,600",092396B,March / July / November,La Trobe 2026 international information,La Trobe official course page,2026-08-20
4,Melbourne,Swinburne University of Technology,Master of Data Science,Yes,Yes,Hawthorn,2,"A$45,010","A$90,020",099117B,2026 intakes,Swinburne 2026 international guide,Swinburne official course page,2026-08-20


In [13]:
print("LIVING-COST DATASET")
print("=" * 60)

print("Columns:")
for i, col in enumerate(living_raw.columns, start=1):
    print(f"{i}. {col}")

print("\nFirst 5 rows:")
display(living_raw.head())

LIVING-COST DATASET
Columns:
1. city
2. numbeo_update_date
3. restaurant_inexpensive
4. restaurant_midrange_two
5. mcdonalds_combo
6. domestic_draft_beer
7. imported_beer
8. cappuccino
9. soft_drink
10. water_033l
11. milk_1l
12. bread_500g
13. rice_1kg
14. eggs_12
15. local_cheese_1kg
16. chicken_1kg
17. beef_1kg
18. apples_1kg
19. bananas_1kg
20. oranges_1kg
21. tomatoes_1kg
22. potatoes_1kg
23. onions_1kg
24. lettuce_1head
25. water_15l
26. wine_midrange
27. domestic_beer_05l
28. imported_beer_033l
29. cigarettes_20
30. transport_one_way
31. transport_monthly_pass
32. taxi_start
33. taxi_per_km
34. taxi_wait_hour
35. gasoline_1l
36. utilities_85sqm
37. mobile_plan
38. internet
39. gym_membership
40. tennis_hour
41. cinema_ticket
42. preschool_monthly
43. international_primary_annual
44. jeans
45. summer_dress
46. nike_running_shoes
47. mens_business_shoes
48. rent_1br_city_centre
49. rent_1br_outside_centre
50. rent_3br_city_centre
51. rent_3br_outside_centre
52. buy_price_sqm_city_

,city,numbeo_update_date,restaurant_inexpensive,restaurant_midrange_two,mcdonalds_combo,domestic_draft_beer,imported_beer,cappuccino,soft_drink,water_033l,milk_1l,bread_500g,rice_1kg,eggs_12,local_cheese_1kg,chicken_1kg,beef_1kg,apples_1kg,bananas_1kg,oranges_1kg,tomatoes_1kg,potatoes_1kg,onions_1kg,lettuce_1head,water_15l,wine_midrange,domestic_beer_05l,imported_beer_033l,cigarettes_20,transport_one_way,transport_monthly_pass,taxi_start,taxi_per_km,taxi_wait_hour,gasoline_1l,utilities_85sqm,mobile_plan,internet,gym_membership,tennis_hour,cinema_ticket,preschool_monthly,international_primary_annual,jeans,summer_dress,nike_running_shoes,mens_business_shoes,rent_1br_city_centre,rent_1br_outside_centre,rent_3br_city_centre,rent_3br_outside_centre,buy_price_sqm_city_centre,buy_price_sqm_outside_centre,average_monthly_net_salary,mortgage_interest_rate,data_status
0,Sydney,15-Aug-2026,25.0,124.0,16.00,10.5,10.0,5.48,3.88,3.12,2.65,3.70,3.36,7.70,17.39,13.73,23.43,5.23,4.36,4.71,6.49,4.32,3.50,3.30,1.72,20.0,5.83,5.85,63.99,5.30,217.39,5.00,2.29,57.32,1.92,319.84,36.00,79.16,102.19,34.82,25.0,3071.85,37576.4,108.47,103.00,152.94,180.36,3517.50,2403.33,6990.00,3987.08,18395.00,11274.69,6010.70,5.81,complete
1,Melbourne,17-Aug-2026,25.0,115.0,17.00,14.0,12.0,5.71,4.43,3.59,2.46,4.61,4.00,8.49,15.14,13.62,21.64,5.95,4.43,4.76,6.92,4.66,3.87,3.56,2.32,19.0,7.61,7.07,63.99,5.50,198.00,5.92,2.00,43.32,1.95,320.71,39.70,79.47,78.79,29.40,25.0,3373.49,45289.3,110.64,69.25,166.56,185.33,2411.13,1979.54,4815.08,3500.73,10026.39,9020.78,6286.11,5.65,complete
2,Brisbane,17-Aug-2026,25.0,120.0,16.00,12.0,12.0,6.20,4.08,3.44,2.50,3.75,3.01,7.22,13.23,12.41,19.85,5.75,4.31,4.25,7.01,4.04,3.35,3.56,1.78,18.0,7.28,6.83,52.50,0.50,30.00,4.30,2.62,58.80,1.93,251.40,42.12,88.24,91.76,30.69,22.5,3052.32,18040.8,119.50,60.77,160.00,154.62,2890.19,2018.44,4969.15,3410.42,11891.06,8759.42,5950.63,6.01,complete
3,Perth,14-Aug-2026,29.0,122.5,15.38,12.0,12.0,6.08,4.05,3.15,2.34,4.33,2.86,7.49,12.42,13.14,19.86,6.34,4.60,4.73,7.47,4.03,2.79,3.26,1.91,18.0,6.22,5.00,59.00,3.50,140.00,5.25,2.04,58.00,1.83,287.69,45.93,89.00,76.71,30.69,25.0,3281.40,33220.0,108.85,93.22,169.21,224.00,2711.62,2272.00,3951.36,3070.00,11887.22,8149.92,6085.66,5.96,complete
4,Adelaide,15-Aug-2026,25.0,120.0,16.00,11.0,12.5,5.74,4.08,3.63,2.93,4.15,3.16,8.07,15.46,13.13,24.92,5.59,4.23,5.20,7.14,4.71,3.46,3.62,3.23,20.0,7.58,8.73,62.00,4.55,120.00,5.00,2.11,44.60,1.87,257.39,35.08,78.50,80.45,19.84,23.0,2955.62,20322.5,103.90,79.95,166.82,191.25,2518.00,1930.00,3504.00,2690.00,14595.38,9915.46,5494.61,6.07,complete


In [14]:
print("EDUCATION COVERAGE")
print("=" * 60)

print("Cities:")
print(sorted(education_raw["City"].dropna().unique()))

print("\nCourses:")
print(sorted(education_raw["Course"].dropna().unique()))

print("\nNumber of universities:")
print(education_raw["University"].nunique())

print("\nLIVING-COST COVERAGE")
print("=" * 60)

print("Cities:")
print(sorted(living_raw["city"].dropna().unique()))

print("\nNumber of cities:")
print(living_raw["city"].nunique())

EDUCATION COVERAGE
Cities:
['Adelaide', 'Brisbane', 'Canberra', 'Darwin', 'Gold Coast', 'Melbourne', 'Perth', 'Sydney']

Courses:
['Master of Business Administration', 'Master of Data Science', 'Master of Information Technology']

Number of universities:
31

LIVING-COST COVERAGE
Cities:
['Adelaide', 'Brisbane', 'Canberra', 'Darwin', 'Gold Coast', 'Melbourne', 'Perth', 'Sydney']

Number of cities:
8


In [15]:
education_cities = set(education_raw["City"].dropna().str.strip())
living_cities = set(living_raw["city"].dropna().str.strip())

print("Cities in education but missing from living data:")
print(sorted(education_cities - living_cities))

print("\nCities in living data but missing from education:")
print(sorted(living_cities - education_cities))

print("\nCity coverage matches:", education_cities == living_cities)

Cities in education but missing from living data:
[]

Cities in living data but missing from education:
[]

City coverage matches: True


In [16]:
course_city_university = (
    education_raw[
        ["City", "Course", "University"]
    ]
    .drop_duplicates()
    .sort_values(["City", "Course", "University"])
)

display(course_city_university)

,City,Course,University
47,Adelaide,Master of Data Science,Adelaide University
48,Adelaide,Master of Data Science,Flinders University
49,Adelaide,Master of Information Technology,Flinders University
44,Brisbane,Master of Business Administration,Griffith University
45,Brisbane,Master of Business Administration,James Cook University
46,Brisbane,Master of Business Administration,Southern Cross University
43,Brisbane,Master of Business Administration,University of Queensland
36,Brisbane,Master of Data Science,Griffith University
35,Brisbane,Master of Data Science,Queensland University of Technology
34,Brisbane,Master of Data Science,University of Queensland


In [17]:
course_city_counts = (
    education_raw
    .groupby(["City", "Course"])["University"]
    .nunique()
    .reset_index(name="University_Count")
    .sort_values(["City", "Course"])
)

display(course_city_counts)

,City,Course,University_Count
0,Adelaide,Master of Data Science,2
1,Adelaide,Master of Information Technology,1
2,Brisbane,Master of Business Administration,4
3,Brisbane,Master of Data Science,3
4,Brisbane,Master of Information Technology,6
5,Canberra,Master of Business Administration,1
6,Canberra,Master of Data Science,1
7,Darwin,Master of Business Administration,1
8,Darwin,Master of Data Science,1
9,Gold Coast,Master of Business Administration,3


In [18]:
education_raw[
    [
        "City",
        "University",
        "Course",
        "International_Student",
        "Campus",
        "Duration_Years",
        "Annual_Tuition_AUD",
        "Total_Tuition_AUD"
    ]
].sort_values(["City", "Course", "University"])

,City,University,Course,International_Student,Campus,Duration_Years,Annual_Tuition_AUD,Total_Tuition_AUD
47,Adelaide,Adelaide University,Master of Data Science,Yes,Adelaide City / Mawson Lakes,2,"A$57,100","A$114,200"
48,Adelaide,Flinders University,Master of Data Science,Yes,City,1 / 2,"A$42,900","A$42,900 / A$85,800"
49,Adelaide,Flinders University,Master of Information Technology,Yes,City,2,"A$42,900","A$85,800"
44,Brisbane,Griffith University,Master of Business Administration,Yes,South Bank,1 / 1.5,Cannot establish exact reliable 2026 figure,Cannot establish exact reliable 2026 figure
45,Brisbane,James Cook University,Master of Business Administration,Yes,Brisbane,1.5,"A$33,133","A$49,699.50"
46,Brisbane,Southern Cross University,Master of Business Administration,Yes,Brisbane,2,"A$26,000","A$52,000"
43,Brisbane,University of Queensland,Master of Business Administration,Yes,Brisbane City,1.5,"A$69,112","A$103,668"
36,Brisbane,Griffith University,Master of Data Science,Yes,Brisbane South (Nathan),1 / 1.5 / 2,Cannot establish exact reliable 2026 figure,Cannot establish exact reliable 2026 figure
35,Brisbane,Queensland University of Technology,Master of Data Science,Yes,Gardens Point,2,"A$44,200","A$88,400"
34,Brisbane,University of Queensland,Master of Data Science,Yes,St Lucia,1.5 / 2,"A$60,952","A$91,428 / A$121,904"


In [19]:
tuition_check = education_raw[
    [
        "City",
        "University",
        "Course",
        "Duration_Years",
        "Annual_Tuition_AUD",
        "Total_Tuition_AUD"
    ]
].copy()

tuition_check["Tuition_Data"] = np.where(
    tuition_check["Annual_Tuition_AUD"].notna(),
    "Available",
    "Cannot establish exact reliable 2026 figure"
)

display(
    tuition_check.sort_values(
        ["City", "Course", "University"]
    )
)

,City,University,Course,Duration_Years,Annual_Tuition_AUD,Total_Tuition_AUD,Tuition_Data
47,Adelaide,Adelaide University,Master of Data Science,2,"A$57,100","A$114,200",Available
48,Adelaide,Flinders University,Master of Data Science,1 / 2,"A$42,900","A$42,900 / A$85,800",Available
49,Adelaide,Flinders University,Master of Information Technology,2,"A$42,900","A$85,800",Available
44,Brisbane,Griffith University,Master of Business Administration,1 / 1.5,Cannot establish exact reliable 2026 figure,Cannot establish exact reliable 2026 figure,Available
45,Brisbane,James Cook University,Master of Business Administration,1.5,"A$33,133","A$49,699.50",Available
46,Brisbane,Southern Cross University,Master of Business Administration,2,"A$26,000","A$52,000",Available
43,Brisbane,University of Queensland,Master of Business Administration,1.5,"A$69,112","A$103,668",Available
36,Brisbane,Griffith University,Master of Data Science,1 / 1.5 / 2,Cannot establish exact reliable 2026 figure,Cannot establish exact reliable 2026 figure,Available
35,Brisbane,Queensland University of Technology,Master of Data Science,2,"A$44,200","A$88,400",Available
34,Brisbane,University of Queensland,Master of Data Science,1.5 / 2,"A$60,952","A$91,428 / A$121,904",Available


In [20]:
education_clean = education_raw.copy()

education_clean["City"] = education_clean["City"].str.strip()
education_clean["University"] = education_clean["University"].str.strip()
education_clean["Course"] = education_clean["Course"].str.strip()

education_clean["Tuition_Status"] = np.where(
    education_clean["Annual_Tuition_AUD"].notna(),
    "Available",
    "Cannot establish exact reliable 2026 figure"
)

education_clean = education_clean.sort_values(
    ["City", "Course", "University"]
).reset_index(drop=True)

display(education_clean)

,City,University,Course,Available,International_Student,Campus,Duration_Years,Annual_Tuition_AUD,Total_Tuition_AUD,CRICOS_Code,Start_Dates,Tuition_Source,Course_Source,Last_Verified,Tuition_Status
0,Adelaide,Adelaide University,Master of Data Science,Yes,Yes,Adelaide City / Mawson Lakes,2,"A$57,100","A$114,200",115904D,2026 intakes,Adelaide University 2026 international fee,Adelaide University official course page,2026-08-20,Available
1,Adelaide,Flinders University,Master of Data Science,Yes,Yes,City,1 / 2,"A$42,900","A$42,900 / A$85,800",105120H / 105119A,March / July,Flinders 2026 fee,Flinders official course page,2026-08-20,Available
2,Adelaide,Flinders University,Master of Information Technology,Yes,Yes,City,2,"A$42,900","A$85,800",038636F,March / July,Flinders 2026 fee,Flinders official course page,2026-08-20,Available
3,Brisbane,Griffith University,Master of Business Administration,Yes,Yes,South Bank,1 / 1.5,Cannot establish exact reliable 2026 figure,Cannot establish exact reliable 2026 figure,011433F,March / July,Griffith exact 2026 international fee not esta...,Griffith official course page,2026-08-20,Available
4,Brisbane,James Cook University,Master of Business Administration,Yes,Yes,Brisbane,1.5,"A$33,133","A$49,699.50",Cannot establish exact reliable 2026 figure,February / May / September,JCU 2026 fee information,JCU official course page,2026-08-20,Available
5,Brisbane,Southern Cross University,Master of Business Administration,Yes,Yes,Brisbane,2,"A$26,000","A$52,000",079662J,June / October,SCU 2026 international course page,SCU official 2026 course page,2026-08-20,Available
6,Brisbane,University of Queensland,Master of Business Administration,Yes,Yes,Brisbane City,1.5,"A$69,112","A$103,668",116551E,February / July,UQ 2026 international fee,UQ official course page,2026-08-20,Available
7,Brisbane,Griffith University,Master of Data Science,Yes,Yes,Brisbane South (Nathan),1 / 1.5 / 2,Cannot establish exact reliable 2026 figure,Cannot establish exact reliable 2026 figure,115627J,March / July,Griffith exact 2026 international fee not esta...,Griffith official course page,2026-08-20,Available
8,Brisbane,Queensland University of Technology,Master of Data Science,Yes,Yes,Gardens Point,2,"A$44,200","A$88,400",116754E,February / July,QUT 2026 international fee,QUT official course page,2026-08-20,Available
9,Brisbane,University of Queensland,Master of Data Science,Yes,Yes,St Lucia,1.5 / 2,"A$60,952","A$91,428 / A$121,904",092454G,February / July,UQ 2026 international fee,UQ official course page,2026-08-20,Available


In [21]:
living_cost_columns = [
    col for col in living_raw.columns
    if any(
        keyword in col.lower()
        for keyword in ["rent", "food", "transport", "utilit"]
    )
]

print("Living-cost columns relevant to GradScope:")
for col in living_cost_columns:
    print("-", col)

Living-cost columns relevant to GradScope:
- transport_one_way
- transport_monthly_pass
- utilities_85sqm
- rent_1br_city_centre
- rent_1br_outside_centre
- rent_3br_city_centre
- rent_3br_outside_centre


In [23]:
education_calc = education_clean.copy()

def clean_money(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    
    try:
        return float(
            value.replace("$", "")
                 .replace(",", "")
                 .replace("AUD", "")
                 .strip()
        )
    except ValueError:
        return np.nan


def duration_range(value):
    if pd.isna(value):
        return (np.nan, np.nan)
    
    value = str(value).strip()
    
    # Handle values such as "1.5 / 2"
    if "/" in value:
        parts = [
            p.strip()
            for p in value.split("/")
        ]
        
        numbers = []
        for p in parts:
            try:
                numbers.append(float(p))
            except ValueError:
                pass
        
        if numbers:
            return (min(numbers), max(numbers))
    
    try:
        number = float(value)
        return (number, number)
    except ValueError:
        return (np.nan, np.nan)


education_calc["Annual_Tuition_Clean_AUD"] = (
    education_calc["Annual_Tuition_AUD"]
    .apply(clean_money)
)

education_calc["Total_Tuition_Clean_AUD"] = (
    education_calc["Total_Tuition_AUD"]
    .apply(clean_money)
)

education_calc[["Duration_Min_Years", "Duration_Max_Years"]] = (
    education_calc["Duration_Years"]
    .apply(lambda x: pd.Series(duration_range(x)))
)

display(
    education_calc[
        [
            "City",
            "University",
            "Course",
            "Duration_Years",
            "Duration_Min_Years",
            "Duration_Max_Years",
            "Annual_Tuition_AUD",
            "Annual_Tuition_Clean_AUD",
            "Tuition_Status"
        ]
    ].head(20)
)

,City,University,Course,Duration_Years,Duration_Min_Years,Duration_Max_Years,Annual_Tuition_AUD,Annual_Tuition_Clean_AUD,Tuition_Status
0,Adelaide,Adelaide University,Master of Data Science,2,2.0,2.0,"A$57,100",NaN,Available
1,Adelaide,Flinders University,Master of Data Science,1 / 2,1.0,2.0,"A$42,900",NaN,Available
2,Adelaide,Flinders University,Master of Information Technology,2,2.0,2.0,"A$42,900",NaN,Available
3,Brisbane,Griffith University,Master of Business Administration,1 / 1.5,1.0,1.5,Cannot establish exact reliable 2026 figure,NaN,Available
4,Brisbane,James Cook University,Master of Business Administration,1.5,1.5,1.5,"A$33,133",NaN,Available
5,Brisbane,Southern Cross University,Master of Business Administration,2,2.0,2.0,"A$26,000",NaN,Available
6,Brisbane,University of Queensland,Master of Business Administration,1.5,1.5,1.5,"A$69,112",NaN,Available
7,Brisbane,Griffith University,Master of Data Science,1 / 1.5 / 2,1.0,2.0,Cannot establish exact reliable 2026 figure,NaN,Available
8,Brisbane,Queensland University of Technology,Master of Data Science,2,2.0,2.0,"A$44,200",NaN,Available
9,Brisbane,University of Queensland,Master of Data Science,1.5 / 2,1.5,2.0,"A$60,952",NaN,Available


In [24]:
def calculate_gradscope_cost(
    course,
    city,
    university,
    scholarship_percent=0
):
    # -----------------------------
    # 1. Find education record
    # -----------------------------
    education_match = education_calc[
        (education_calc["Course"] == course) &
        (education_calc["City"] == city) &
        (education_calc["University"] == university)
    ]

    if education_match.empty:
        return {
            "status": "not_found",
            "message": "This course/university combination was not found."
        }

    education_row = education_match.iloc[0]

    # -----------------------------
    # 2. Find city living costs
    # -----------------------------
    living_match = living_raw[
        living_raw["city"].astype(str).str.strip() == city
    ]

    if living_match.empty:
        return {
            "status": "missing_city_cost",
            "message": "Living-cost data is unavailable for this city."
        }

    living_row = living_match.iloc[0]

    # -----------------------------
    # 3. Tuition
    # -----------------------------
    annual_tuition = education_row["Annual_Tuition_Clean_AUD"]

    duration_min = education_row["Duration_Min_Years"]
    duration_max = education_row["Duration_Max_Years"]

    tuition_available = pd.notna(annual_tuition)

    # -----------------------------
    # 4. Scholarship
    # -----------------------------
    scholarship_percent = max(
        0,
        min(float(scholarship_percent), 100)
    )

    # -----------------------------
    # 5. Living costs
    # -----------------------------
    monthly_rent = float(living_row["monthly_rent_cost"])
    monthly_food = float(living_row["monthly_food_cost"])
    monthly_transport = float(living_row["monthly_transport_cost"])
    monthly_utilities = float(living_row["monthly_utilities_cost"])

    monthly_living = (
        monthly_rent
        + monthly_food
        + monthly_transport
        + monthly_utilities
    )

    annual_living = monthly_living * 12

    # -----------------------------
    # 6. Tuition calculation
    # -----------------------------
    if tuition_available and pd.notna(duration_min):
        
        tuition_min = annual_tuition * duration_min
        tuition_max = annual_tuition * duration_max

        scholarship_min = (
            tuition_min * scholarship_percent / 100
        )

        scholarship_max = (
            tuition_max * scholarship_percent / 100
        )

        remaining_tuition_min = (
            tuition_min - scholarship_min
        )

        remaining_tuition_max = (
            tuition_max - scholarship_max
        )

        total_cost_min = (
            remaining_tuition_min
            + annual_living * duration_min
        )

        total_cost_max = (
            remaining_tuition_max
            + annual_living * duration_max
        )

    else:
        # Tuition deliberately excluded
        # when exact reliable 2026 tuition isn't available.
        remaining_tuition_min = np.nan
        remaining_tuition_max = np.nan

        total_cost_min = annual_living * duration_min
        total_cost_max = annual_living * duration_max

    return {
        "status": "success",

        "course": course,
        "city": city,
        "university": university,

        "duration_min_years": duration_min,
        "duration_max_years": duration_max,

        "annual_tuition_aud": annual_tuition,

        "scholarship_percent": scholarship_percent,

        "monthly_rent_aud": monthly_rent,
        "monthly_food_aud": monthly_food,
        "monthly_transport_aud": monthly_transport,
        "monthly_utilities_aud": monthly_utilities,

        "monthly_living_aud": monthly_living,
        "annual_living_aud": annual_living,

        "remaining_tuition_min_aud": remaining_tuition_min,
        "remaining_tuition_max_aud": remaining_tuition_max,

        "total_cost_min_aud": total_cost_min,
        "total_cost_max_aud": total_cost_max,

        "tuition_included": tuition_available
    }

print("GradScope calculation engine created.")

GradScope calculation engine created.


In [25]:
print(living_raw.columns.tolist())

['city', 'numbeo_update_date', 'restaurant_inexpensive', 'restaurant_midrange_two', 'mcdonalds_combo', 'domestic_draft_beer', 'imported_beer', 'cappuccino', 'soft_drink', 'water_033l', 'milk_1l', 'bread_500g', 'rice_1kg', 'eggs_12', 'local_cheese_1kg', 'chicken_1kg', 'beef_1kg', 'apples_1kg', 'bananas_1kg', 'oranges_1kg', 'tomatoes_1kg', 'potatoes_1kg', 'onions_1kg', 'lettuce_1head', 'water_15l', 'wine_midrange', 'domestic_beer_05l', 'imported_beer_033l', 'cigarettes_20', 'transport_one_way', 'transport_monthly_pass', 'taxi_start', 'taxi_per_km', 'taxi_wait_hour', 'gasoline_1l', 'utilities_85sqm', 'mobile_plan', 'internet', 'gym_membership', 'tennis_hour', 'cinema_ticket', 'preschool_monthly', 'international_primary_annual', 'jeans', 'summer_dress', 'nike_running_shoes', 'mens_business_shoes', 'rent_1br_city_centre', 'rent_1br_outside_centre', 'rent_3br_city_centre', 'rent_3br_outside_centre', 'buy_price_sqm_city_centre', 'buy_price_sqm_outside_centre', 'average_monthly_net_salary', 'm

In [26]:
required_living_columns = [
    "city",
    "monthly_rent_cost",
    "monthly_food_cost",
    "monthly_transport_cost",
    "monthly_utilities_cost"
]

print("Checking required GradScope columns:\n")

for col in required_living_columns:
    print(f"{col}: {'FOUND' if col in living_raw.columns else 'MISSING'}")

Checking required GradScope columns:

city: FOUND
monthly_rent_cost: MISSING
monthly_food_cost: MISSING
monthly_transport_cost: MISSING
monthly_utilities_cost: MISSING


In [27]:
for col in living_raw.columns:
    col_lower = col.lower()
    
    if any(word in col_lower for word in [
        "rent",
        "food",
        "transport",
        "transportation",
        "utility",
        "utilities"
    ]):
        print(col)

transport_one_way
transport_monthly_pass
utilities_85sqm
rent_1br_city_centre
rent_1br_outside_centre
rent_3br_city_centre
rent_3br_outside_centre


In [28]:
food_columns = [
    col for col in living_raw.columns
    if any(word in col.lower() for word in [
        "food",
        "grocery",
        "grocer",
        "bread",
        "milk",
        "rice",
        "egg",
        "chicken",
        "beef",
        "meal",
        "restaurant"
    ])
]

print("Possible food-related columns:")
for col in food_columns:
    print("-", col)

Possible food-related columns:
- restaurant_inexpensive
- restaurant_midrange_two
- milk_1l
- bread_500g
- rice_1kg
- eggs_12
- chicken_1kg
- beef_1kg
- buy_price_sqm_city_centre
- buy_price_sqm_outside_centre


In [29]:
print("Food basket source columns:")
for col in [
    "milk_1l",
    "bread_500g",
    "rice_1kg",
    "eggs_12",
    "chicken_1kg",
    "beef_1kg"
]:
    print(f"{col}: {'FOUND' if col in living_raw.columns else 'MISSING'}")

Food basket source columns:
milk_1l: FOUND
bread_500g: FOUND
rice_1kg: FOUND
eggs_12: FOUND
chicken_1kg: FOUND
beef_1kg: FOUND


In [30]:
# CELL 22 — GradScope monthly food basket
# Quantities are based on the existing GradScope methodology.

FOOD_BASKET = {
    "Milk (1L)": {
        "column": "milk_1l",
        "quantity": 8
    },
    "Bread (500g)": {
        "column": "bread_500g",
        "quantity": 8
    },
    "Rice (1kg)": {
        "column": "rice_1kg",
        "quantity": 4
    },
    "Eggs (12)": {
        "column": "eggs_12",
        "quantity": 2
    },
    "Chicken (1kg)": {
        "column": "chicken_1kg",
        "quantity": 2
    },
    "Beef (1kg)": {
        "column": "beef_1kg",
        "quantity": 1
    }
}

print("GradScope food basket:")
for item, config in FOOD_BASKET.items():
    print(f"- {item}: {config['quantity']} × {config['column']}")

print("\nFood basket source columns:")
for item, config in FOOD_BASKET.items():
    column = config["column"]
    print(f"{column}: {'FOUND' if column in living_raw.columns else 'MISSING'}")

GradScope food basket:
- Milk (1L): 8 × milk_1l
- Bread (500g): 8 × bread_500g
- Rice (1kg): 4 × rice_1kg
- Eggs (12): 2 × eggs_12
- Chicken (1kg): 2 × chicken_1kg
- Beef (1kg): 1 × beef_1kg

Food basket source columns:
milk_1l: FOUND
bread_500g: FOUND
rice_1kg: FOUND
eggs_12: FOUND
chicken_1kg: FOUND
beef_1kg: FOUND


In [31]:
# CELL 23 — Calculate monthly food basket cost

def calculate_monthly_food_cost(city):
    row = living_raw[living_raw["city"] == city]

    if row.empty:
        raise ValueError(f"City not found: {city}")

    row = row.iloc[0]

    total = 0

    for config in FOOD_BASKET.values():
        column = config["column"]
        quantity = config["quantity"]

        value = pd.to_numeric(row[column], errors="coerce")

        if pd.isna(value):
            return None

        total += value * quantity

    return round(total, 2)


# Test with all GradScope cities
cities = sorted(living_raw["city"].dropna().unique())

food_costs = []

for city in cities:
    cost = calculate_monthly_food_cost(city)

    food_costs.append({
        "City": city,
        "Monthly_Food_Cost_AUD": cost
    })

food_costs_df = pd.DataFrame(food_costs)

display(food_costs_df)

,City,Monthly_Food_Cost_AUD
0,Adelaide,136.60
1,Brisbane,121.15
2,Canberra,139.36
3,Darwin,114.68
4,Gold Coast,124.88
5,Melbourne,138.42
6,Perth,125.92
7,Sydney,130.53


In [32]:
# CELL 24 — Build GradScope living-cost model

living_cost_df = living_raw[[
    "city",
    "rent_1br_outside_centre",
    "transport_monthly_pass",
    "utilities_85sqm",
]].copy()

living_cost_df = living_cost_df.merge(
    food_costs_df,
    left_on="city",
    right_on="City",
    how="left"
)

living_cost_df = living_cost_df.rename(columns={
    "rent_1br_outside_centre": "Monthly_Rent_AUD",
    "transport_monthly_pass": "Monthly_Transport_AUD",
    "utilities_85sqm": "Monthly_Utilities_AUD"
})

living_cost_df = living_cost_df[[
    "city",
    "Monthly_Rent_AUD",
    "Monthly_Food_Cost_AUD",
    "Monthly_Transport_AUD",
    "Monthly_Utilities_AUD"
]]

display(living_cost_df)

,city,Monthly_Rent_AUD,Monthly_Food_Cost_AUD,Monthly_Transport_AUD,Monthly_Utilities_AUD
0,Sydney,2403.33,130.53,217.39,319.84
1,Melbourne,1979.54,138.42,198.00,320.71
2,Brisbane,2018.44,121.15,30.00,251.40
3,Perth,2272.00,125.92,140.00,287.69
4,Adelaide,1930.00,136.60,120.00,257.39
5,Canberra,2111.82,139.36,132.80,263.39
6,Gold Coast,2127.50,124.88,80.00,311.81
7,Darwin,2426.67,114.68,69.69,311.56


In [33]:
# CELL 25 — Calculate total monthly living cost

living_cost_df["Monthly_Living_Cost_AUD"] = (
    living_cost_df["Monthly_Rent_AUD"]
    + living_cost_df["Monthly_Food_Cost_AUD"]
    + living_cost_df["Monthly_Transport_AUD"]
    + living_cost_df["Monthly_Utilities_AUD"]
)

display(
    living_cost_df[
        [
            "city",
            "Monthly_Rent_AUD",
            "Monthly_Food_Cost_AUD",
            "Monthly_Transport_AUD",
            "Monthly_Utilities_AUD",
            "Monthly_Living_Cost_AUD"
        ]
    ]
)

,city,Monthly_Rent_AUD,Monthly_Food_Cost_AUD,Monthly_Transport_AUD,Monthly_Utilities_AUD,Monthly_Living_Cost_AUD
0,Sydney,2403.33,130.53,217.39,319.84,3071.09
1,Melbourne,1979.54,138.42,198.00,320.71,2636.67
2,Brisbane,2018.44,121.15,30.00,251.40,2420.99
3,Perth,2272.00,125.92,140.00,287.69,2825.61
4,Adelaide,1930.00,136.60,120.00,257.39,2443.99
5,Canberra,2111.82,139.36,132.80,263.39,2647.37
6,Gold Coast,2127.50,124.88,80.00,311.81,2644.19
7,Darwin,2426.67,114.68,69.69,311.56,2922.60


In [37]:
# CELL 26 — Robust GradScope final cost calculation

import re

def parse_duration(value):
    """
    Convert different duration formats into minimum and maximum years.
    Examples:
    2
    1.5 / 2
    1.5/2
    2 years
    """

    text = str(value).strip()

    numbers = re.findall(r"\d+(?:\.\d+)?", text)

    if not numbers:
        raise ValueError(f"Cannot determine course duration from: {value}")

    numbers = [float(x) for x in numbers]

    return min(numbers), max(numbers)


def parse_tuition(value):
    """
    Convert tuition values such as:
    A$44,160
    $44,160
    44160
    into a numeric AUD value.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    if "Cannot establish" in text:
        return None

    numbers = re.findall(r"\d+(?:,\d{3})*(?:\.\d+)?", text)

    if not numbers:
        return None

    return float(numbers[0].replace(",", ""))


def calculate_gradscope_cost(
    city,
    university,
    course,
    scholarship_percent=0
):

    # Find selected university/course
    education_match = education_raw[
        (education_raw["City"] == city) &
        (education_raw["University"] == university) &
        (education_raw["Course"] == course)
    ]

    if education_match.empty:
        raise ValueError(
            f"No matching record found for {course} at {university} in {city}"
        )

    education = education_match.iloc[0]

    # -----------------------------
    # Duration
    # -----------------------------

    duration_min, duration_max = parse_duration(
        education["Duration_Years"]
    )

    # -----------------------------
    # Tuition
    # -----------------------------

    annual_tuition = parse_tuition(
        education["Annual_Tuition_AUD"]
    )

    if annual_tuition is not None:

        total_tuition_min = annual_tuition * duration_min
        total_tuition_max = annual_tuition * duration_max

        scholarship_factor = 1 - (
            scholarship_percent / 100
        )

        remaining_tuition_min = (
            total_tuition_min * scholarship_factor
        )

        remaining_tuition_max = (
            total_tuition_max * scholarship_factor
        )

    else:

        total_tuition_min = None
        total_tuition_max = None

        remaining_tuition_min = None
        remaining_tuition_max = None

    # -----------------------------
    # Living cost
    # -----------------------------

    living_match = living_cost_df[
        living_cost_df["city"] == city
    ]

    if living_match.empty:
        raise ValueError(
            f"Living-cost data not found for {city}"
        )

    monthly_living = float(
        living_match.iloc[0]["Monthly_Living_Cost_AUD"]
    )

    living_cost_min = (
        monthly_living * 12 * duration_min
    )

    living_cost_max = (
        monthly_living * 12 * duration_max
    )

    # -----------------------------
    # Final estimated cost
    # -----------------------------

    if remaining_tuition_min is not None:

        total_cost_min = (
            remaining_tuition_min +
            living_cost_min
        )

        total_cost_max = (
            remaining_tuition_max +
            living_cost_max
        )

    else:

        total_cost_min = None
        total_cost_max = None

    return {
        "city": city,
        "university": university,
        "course": course,

        "duration_min_years": duration_min,
        "duration_max_years": duration_max,

        "annual_tuition_aud": annual_tuition,

        "scholarship_percent": scholarship_percent,

        "remaining_tuition_min_aud":
            remaining_tuition_min,

        "remaining_tuition_max_aud":
            remaining_tuition_max,

        "monthly_living_cost_aud":
            monthly_living,

        "living_cost_min_aud":
            living_cost_min,

        "living_cost_max_aud":
            living_cost_max,

        "total_cost_min_aud":
            total_cost_min,

        "total_cost_max_aud":
            total_cost_max
    }


print("GradScope final calculation engine ready.")

GradScope final calculation engine ready.


In [35]:
# CELL 27 — Test GradScope calculation with a real university

test_result = calculate_gradscope_cost(
    city="Melbourne",
    university="RMIT University",
    course="Master of Data Science",
    scholarship_percent=20
)

test_result

{'city': 'Melbourne',
 'university': 'RMIT University',
 'course': 'Master of Data Science',
 'duration_min_years': 2.0,
 'duration_max_years': 2.0,
 'annual_tuition_aud': 44160.0,
 'scholarship_percent': 20,
 'remaining_tuition_min_aud': 70656.0,
 'remaining_tuition_max_aud': 70656.0,
 'monthly_living_cost_aud': 2636.67,
 'living_cost_min_aud': 63280.08,
 'living_cost_max_aud': 63280.08,
 'total_cost_min_aud': 133936.08000000002,
 'total_cost_max_aud': 133936.08000000002}

In [39]:
# CELL 28 — FINAL GRADSCOPE CALCULATIONS FOR ALL UNIVERSITIES

calculator_records = []
calculation_errors = []

for _, row in education_raw.iterrows():

    try:
        result = calculate_gradscope_cost(
            city=row["City"],
            university=row["University"],
            course=row["Course"],
            scholarship_percent=0
        )

        result["tuition_available"] = (
            pd.notna(row["Annual_Tuition_AUD"])
            and "Cannot establish" not in str(row["Annual_Tuition_AUD"])
        )

        calculator_records.append(result)

    except Exception as e:
        calculation_errors.append({
            "city": row["City"],
            "university": row["University"],
            "course": row["Course"],
            "error": str(e)
        })

calculator_df = pd.DataFrame(calculator_records)

print("FINAL GRADSCOPE CALCULATION")
print("=" * 50)
print("Successful records:", len(calculator_df))
print("Failed records:", len(calculation_errors))

if calculation_errors:
    print("\nFailed calculations:")
    display(pd.DataFrame(calculation_errors))
else:
    print("All university/course combinations calculated successfully.")

display(calculator_df.head(10))

FINAL GRADSCOPE CALCULATION
Successful records: 68
Failed records: 1

Failed calculations:


,city,university,course,error
0,Gold Coast,Griffith University,Master of Business Administration,Cannot determine course duration from: Variable


,city,university,course,duration_min_years,duration_max_years,annual_tuition_aud,scholarship_percent,remaining_tuition_min_aud,remaining_tuition_max_aud,monthly_living_cost_aud,living_cost_min_aud,living_cost_max_aud,total_cost_min_aud,total_cost_max_aud,tuition_available
0,Melbourne,RMIT University,Master of Data Science,2.0,2.0,44160.0,0,88320.0,88320.0,2636.67,63280.08,63280.08,151600.08,151600.08,True
1,Melbourne,The University of Melbourne,Master of Data Science,2.0,2.0,57984.0,0,115968.0,115968.0,2636.67,63280.08,63280.08,179248.08,179248.08,True
2,Melbourne,Monash University,Master of Data Science,2.0,2.0,52900.0,0,105800.0,105800.0,2636.67,63280.08,63280.08,169080.08,169080.08,True
3,Melbourne,La Trobe University,Master of Data Science,2.0,2.0,43800.0,0,87600.0,87600.0,2636.67,63280.08,63280.08,150880.08,150880.08,True
4,Melbourne,Swinburne University of Technology,Master of Data Science,2.0,2.0,45010.0,0,90020.0,90020.0,2636.67,63280.08,63280.08,153300.08,153300.08,True
5,Melbourne,Federation University Australia,Master of Data Science,2.0,2.0,41400.0,0,82800.0,82800.0,2636.67,63280.08,63280.08,146080.08,146080.08,True
6,Melbourne,RMIT University,Master of Information Technology,2.0,2.0,44160.0,0,88320.0,88320.0,2636.67,63280.08,63280.08,151600.08,151600.08,True
7,Melbourne,Deakin University,Master of Information Technology,2.0,2.0,44200.0,0,88400.0,88400.0,2636.67,63280.08,63280.08,151680.08,151680.08,True
8,Melbourne,Monash University,Master of Information Technology,2.0,2.0,NaN,0,NaN,NaN,2636.67,63280.08,63280.08,NaN,NaN,False
9,Melbourne,La Trobe University,Master of Information Technology,1.5,2.0,NaN,0,NaN,NaN,2636.67,47460.06,63280.08,NaN,NaN,False


In [40]:
# CELL 29 — Prepare final GradScope calculator dataset

calculator_df = calculator_df[
    [
        "city",
        "university",
        "course",
        "duration_min_years",
        "duration_max_years",
        "annual_tuition_aud",
        "scholarship_percent",
        "remaining_tuition_min_aud",
        "remaining_tuition_max_aud",
        "monthly_living_cost_aud",
        "living_cost_min_aud",
        "living_cost_max_aud",
        "total_cost_min_aud",
        "total_cost_max_aud",
        "tuition_available"
    ]
].copy()

calculator_df = calculator_df.round(2)

print("FINAL GRADSCOPE CALCULATOR DATASET")
print("=" * 50)
print("Records:", len(calculator_df))
print("Columns:", len(calculator_df.columns))

display(calculator_df.head(10))

FINAL GRADSCOPE CALCULATOR DATASET
Records: 68
Columns: 15


,city,university,course,duration_min_years,duration_max_years,annual_tuition_aud,scholarship_percent,remaining_tuition_min_aud,remaining_tuition_max_aud,monthly_living_cost_aud,living_cost_min_aud,living_cost_max_aud,total_cost_min_aud,total_cost_max_aud,tuition_available
0,Melbourne,RMIT University,Master of Data Science,2.0,2.0,44160.0,0,88320.0,88320.0,2636.67,63280.08,63280.08,151600.08,151600.08,True
1,Melbourne,The University of Melbourne,Master of Data Science,2.0,2.0,57984.0,0,115968.0,115968.0,2636.67,63280.08,63280.08,179248.08,179248.08,True
2,Melbourne,Monash University,Master of Data Science,2.0,2.0,52900.0,0,105800.0,105800.0,2636.67,63280.08,63280.08,169080.08,169080.08,True
3,Melbourne,La Trobe University,Master of Data Science,2.0,2.0,43800.0,0,87600.0,87600.0,2636.67,63280.08,63280.08,150880.08,150880.08,True
4,Melbourne,Swinburne University of Technology,Master of Data Science,2.0,2.0,45010.0,0,90020.0,90020.0,2636.67,63280.08,63280.08,153300.08,153300.08,True
5,Melbourne,Federation University Australia,Master of Data Science,2.0,2.0,41400.0,0,82800.0,82800.0,2636.67,63280.08,63280.08,146080.08,146080.08,True
6,Melbourne,RMIT University,Master of Information Technology,2.0,2.0,44160.0,0,88320.0,88320.0,2636.67,63280.08,63280.08,151600.08,151600.08,True
7,Melbourne,Deakin University,Master of Information Technology,2.0,2.0,44200.0,0,88400.0,88400.0,2636.67,63280.08,63280.08,151680.08,151680.08,True
8,Melbourne,Monash University,Master of Information Technology,2.0,2.0,NaN,0,NaN,NaN,2636.67,63280.08,63280.08,NaN,NaN,False
9,Melbourne,La Trobe University,Master of Information Technology,1.5,2.0,NaN,0,NaN,NaN,2636.67,47460.06,63280.08,NaN,NaN,False


In [41]:
# CELL 30 — Validate final calculator dataset

print("VALIDATION")
print("=" * 50)

print("Total records:", len(calculator_df))
print("Unique cities:", calculator_df["city"].nunique())
print("Unique universities:", calculator_df["university"].nunique())
print("Unique courses:", calculator_df["course"].nunique())

print("\nTuition availability:")
print(calculator_df["tuition_available"].value_counts(dropna=False))

print("\nMissing values:")
display(calculator_df.isna().sum())

VALIDATION
Total records: 68
Unique cities: 8
Unique universities: 31
Unique courses: 3

Tuition availability:
tuition_available
True     55
False    13
Name: count, dtype: int64

Missing values:


city                          0
university                    0
course                        0
duration_min_years            0
duration_max_years            0
annual_tuition_aud           13
scholarship_percent           0
remaining_tuition_min_aud    13
remaining_tuition_max_aud    13
monthly_living_cost_aud       0
living_cost_min_aud           0
living_cost_max_aud           0
total_cost_min_aud           13
total_cost_max_aud           13
tuition_available             0
dtype: int64

In [42]:
# CELL 31 — Export final GradScope calculator dataset

OUTPUT_FILE = PROCESSED_DIR / "gradscope_calculator_2026.csv"

calculator_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("GradScope calculator dataset exported successfully.")
print("File:", OUTPUT_FILE)
print("Records:", len(calculator_df))

GradScope calculator dataset exported successfully.
File: ../data/processed/gradscope_calculator_2026.csv
Records: 68
